# 8 — Estrazione embedding testuali del vocabolario (concept naming)

Notebook per la fase di **concept naming** del MSAE.

> **Nota — indipendente dal modello MSAE.** Questo notebook produce gli embedding CLIP
> **testuali** del vocabolario (768D), che dipendono solo da CLIP e **non** dalla dimensione
> latente del SAE. Lo stesso file di output è quindi condiviso da entrambi i modelli
> (expansion 8 → latent 6144 ed expansion 16 → latent 12288): **non serve** scegliere il
> modello qui. La selezione `EXPANSION_FACTOR` compare nei notebook 9, 10 e 11, che caricano
> il checkpoint / le attivazioni del SAE.

**Cosa fa**: estrae gli embedding CLIP (CommonPool XL ViT-L/14, 768D) per un vocabolario
**combinato** di attributi:
- **SUN Attributes** (102 attributi di scena — Patterson & Hays, CVPR 2012) da `labels_sunattribute.txt`
- **attribute_index.json** (620 attributi visivi: colori, materiali, forme, texture, azioni)

I due insiemi vengono uniti e **deduplicati** (i termini esatti in comune — es. `brick`,
`sunny`, `glass`, `dirty` — vengono tenuti una sola volta).

**Perché**: nello step successivo ogni feature del decoder MSAE viene "nominata" trovando
il termine del vocabolario con maggior cosine similarity rispetto alla direzione della feature
nello spazio CLIP.


## 1. Setup — Drive

Monta Google Drive. Non è necessario clonare la repo MSAE: i vocabolari (`labels_sunattribute.txt`
e `attribute_index.json`) vengono letti direttamente da Drive.

In [ ]:
from google.colab import drive
import pathlib

try:
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError(f"Mount Drive fallito: {e}")

DRIVE = pathlib.Path("/content/drive/MyDrive/tesi")
if not DRIVE.exists():
    raise FileNotFoundError(f"Cartella Drive non trovata: {DRIVE}")
print(f"Drive OK: {DRIVE}")

# Due sorgenti di vocabolario: SUN attributes (txt) + attribute_index (json)
VOCAB_FILE = DRIVE / "labels_sunattribute.txt"
ATTR_JSON  = DRIVE / "attribute_index.json"
for p in [VOCAB_FILE, ATTR_JSON]:
    if not p.exists():
        raise FileNotFoundError(
            f"File vocabolario non trovato: {p}\n"
            "Carica labels_sunattribute.txt e attribute_index.json in MyDrive/tesi/ prima di eseguire."
        )
print(f"Vocab SUN OK : {VOCAB_FILE}")
print(f"Vocab JSON OK: {ATTR_JSON}")

## 2. Installazione di open_clip e caricamento del modello

Installa `open_clip_torch` e carica **CLIP-ViT-L-14-CommonPool.XL** (stesso modello usato
per estrarre gli embedding delle immagini nel notebook 6, quindi lo spazio vettoriale è
coerente con il decoder MSAE).

In [ ]:
!pip install open_clip_torch -q

import sys
import open_clip
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    print("ATTENZIONE: GPU non rilevata. Cambia runtime a T4 (Runtime -> Change runtime type -> GPU).")

model, _, _ = open_clip.create_model_and_transforms(
    'hf-hub:laion/CLIP-ViT-L-14-CommonPool.XL.clip-s13B-b90K'
)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer('hf-hub:laion/CLIP-ViT-L-14-CommonPool.XL.clip-s13B-b90K')

print("Modello caricato: CLIP-ViT-L-14-CommonPool.XL (embedding dim = 768)")

## 3. Caricamento e merge dei vocabolari

Unisce due sorgenti di attributi:
1. **SUN Attributes** da `labels_sunattribute.txt` (102 attributi di scena)
2. **attribute_index.json** (620 attributi visivi: colori, materiali, forme, texture, azioni)

I termini vengono codificati **grezzi** (senza template tipo `"a photo of ..."`): i due
vocabolari sono eterogenei (nomi, gerundi, aggettivi) e nessun template unico è
grammaticalmente corretto per tutte le categorie. Il merge **deduplica** i termini esatti
in comune (es. `brick`, `sunny`, `glass`, `dirty`), preservando l'ordine: prima SUN, poi i
nuovi termini del json.

In [ ]:
import json

# 1. SUN Attributes (scena: attività, materiali, illuminazione, layout, mood)
with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
    sun_terms = [line.strip() for line in f if line.strip()]

# 2. attribute_index.json: le chiavi sono i termini (i valori sono indici, ignorati)
with open(ATTR_JSON, 'r', encoding='utf-8') as f:
    attr_index = json.load(f)
attr_terms = [k.strip() for k in attr_index.keys() if k.strip()]

# 3. Merge con dedup case-insensitive, preservando l'ordine: prima SUN, poi i nuovi
#    termini del json. Evita doppioni esatti (es. "brick", "sunny", "glass", "dirty").
seen = set()
terms = []
for t in sun_terms + attr_terms:
    key = t.lower()
    if key not in seen:
        seen.add(key)
        terms.append(t)

N = len(terms)
n_overlap = len(sun_terms) + len(attr_terms) - N
print(f"SUN attributes        : {len(sun_terms)}")
print(f"attribute_index.json  : {len(attr_terms)}")
print(f"Termini duplicati      : {n_overlap}")
print(f"Vocabolario combinato : {N} termini")
print(f"Primi 5 : {terms[:5]}")
print(f"Ultimi 5: {terms[-5:]}")

## 4. Generazione degli embedding CLIP

Per ogni termine del vocabolario:
1. **Tokenizzazione**: `tokenizer(term, truncate=True)` → tensore di indici
2. **Encoding**: `model.encode_text(tokens)` in autocast fp16 → vettore 768D
3. **L2-normalizzazione**: norma unitaria per ogni vettore

Con 102 termini, il batch unico è sufficiente.

In [ ]:
BATCH_SIZE = 512
D = 768

embeddings = np.zeros((N, D), dtype=np.float32)

print("Generazione embedding in corso...")
for start in tqdm(range(0, N, BATCH_SIZE), desc="Batch"):
    end = min(start + BATCH_SIZE, N)
    batch_terms = terms[start:end]

    tokens = tokenizer(batch_terms).to(device)

    with torch.no_grad():
        text_features = model.encode_text(tokens)

    text_features = text_features.float()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    embeddings[start:end] = text_features.cpu().numpy()

print(f"\nEstrazione completata. Shape finale: {embeddings.shape}")

## 5. Salvataggio su Drive e sanity check

Salva due file allineati su Drive (prefisso `VOCAB_NAME`):

- **`{VOCAB_NAME}_ViT-L~14_text_{N}_{D}.npy`** — embedding in formato raw memmap.
  Il naming segue la convenzione `SAEDataset` della repo MSAE (`..._{N}_{D}.npy`).
- **`{VOCAB_NAME}_ViT-L~14_text_{N}.txt`** — termini in ordine identico agli embedding
  (indice riga = indice embedding). Termini **bare**, etichette leggibili nella concept map.

`N` (numero termini dopo dedup) è calcolato dinamicamente, così cambiando vocabolario il
nome file si aggiorna da solo. Il notebook 9 ritrova il file via glob sul prefisso `VOCAB_NAME`.

In [ ]:
OUT_DIR = DRIVE
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefisso del vocabolario: cambialo se cambi sorgenti. N e D sono dedotti.
VOCAB_NAME = "sun_attrindex"

EMB_PATH   = OUT_DIR / f"{VOCAB_NAME}_ViT-L~14_text_{N}_{D}.npy"
TERMS_PATH = OUT_DIR / f"{VOCAB_NAME}_ViT-L~14_text_{N}.txt"

mm = np.memmap(EMB_PATH, dtype=np.float32, mode='w+', shape=embeddings.shape)
mm[:] = embeddings
mm.flush()
del mm

# Salva i termini bare (non i prompt) per etichette leggibili nella concept map
with open(TERMS_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(terms))

# --- Sanity check ---
print(f"Shape embedding  : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")
norm_mean = np.linalg.norm(embeddings, axis=1).mean()
ok = "OK" if abs(norm_mean - 1.0) < 1e-3 else "FAIL"
print(f"Norma media      : {norm_mean:.4f}  [{ok}]")
print(f"\nPrimi 10 termini con i primi 3 valori dell'embedding:")
for i in range(min(10, N)):
    print(f"  {terms[i]:>30s} -> {embeddings[i, :3]}")
print()
print("File salvati:")
print(f"  - {EMB_PATH}  ({EMB_PATH.stat().st_size / 1e3:.1f} KB)")
print(f"  - {TERMS_PATH}  ({TERMS_PATH.stat().st_size / 1e3:.1f} KB)")